In [0]:
from pyspark.sql.functions import *
import time

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@strdatabricks.dfs.core.windows.net/customers")

df.show()

+-----------+----------+---------+--------------------+------------------+-----+-------------+
|customer_id|first_name|last_name|               email|              city|state|_rescued_data|
+-----------+----------+---------+--------------------+------------------+-----+-------------+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|         NULL|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|         NULL|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|         NULL|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|         NULL|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|         NULL|
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|     North Matthew|   IN|         NULL|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|        Joneshaven|   VA|         NULL|
|     C00008|      Paul|   Campos|sullivanjeremy@h

In [0]:
df = df.drop("_rescued_data")
df.show()

+-----------+----------+---------+--------------------+------------------+-----+
|customer_id|first_name|last_name|               email|              city|state|
+-----------+----------+---------+--------------------+------------------+-----+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|     North Matthew|   IN|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|        Joneshaven|   VA|
|     C00008|      Paul|   Campos|sullivanjeremy@ho...|  South Nathanfurt|   CT|
|     C00009|      Mary|    Green|  dennis03@yahoo.com|      Kimberlyview|   MD|
|     C00010|     James|    

In [0]:
df = df.withColumn(
    "domain", split(col("email"),"@").getItem(1)
)
df.show()

+-----------+----------+---------+--------------------+------------------+-----+------------------+
|customer_id|first_name|last_name|               email|              city|state|            domain|
+-----------+----------+---------+--------------------+------------------+-----+------------------+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|          ryan.org|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|         kelly.com|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|         yahoo.com|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|     campbell.info|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|         yahoo.com|
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|     North Matthew|   IN|         gmail.com|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|        Joneshaven|   VA|         gmail.com|


In [0]:
df.groupBy("domain").agg(count("customer_id").alias("total_customers")).sort("total_customers", ascending=False).show(10)

+-------------+---------------+
|       domain|total_customers|
+-------------+---------------+
|    gmail.com|            374|
|  hotmail.com|            360|
|    yahoo.com|            331|
|    davis.com|              8|
|    brown.com|              8|
|    smith.com|              7|
|  johnson.com|              5|
|hernandez.com|              5|
|  kennedy.com|              4|
|   miller.com|              4|
+-------------+---------------+
only showing top 10 rows


In [0]:
df_gmail = df.filter(col("domain") == "gmail.com")
df_gmail.show(10)
df_yahoo = df.filter(col("domain") == "yahoo.com")
df_yahoo.show(10)
df_hotmail = df.filter(col("domain") == "hotmail.com")
df_hotmail.show(10)



+-----------+----------+---------+--------------------+-----------------+-----+---------+
|customer_id|first_name|last_name|               email|             city|state|   domain|
+-----------+----------+---------+--------------------+-----------------+-----+---------+
|     C00006|     Kevin| Mccarthy|traceyramos@gmail...|    North Matthew|   IN|gmail.com|
|     C00007|    Amanda|    Doyle|scottallen@gmail.com|       Joneshaven|   VA|gmail.com|
|     C00011|     Jacob|       Le|   anita65@gmail.com|      Houstonfurt|   AR|gmail.com|
|     C00012|      Chad|    Banks|beardtravis@gmail...|East Jenniferview|   WI|gmail.com|
|     C00013|     James|   Martin|     jwood@gmail.com|Lake Gregoryshire|   OR|gmail.com|
|     C00017|    Angela|   Carter|rhondaferguson@gm...| West Loriborough|   GA|gmail.com|
|     C00028|     Barry|    Baker|  steven27@gmail.com|North Gregoryfurt|   NM|gmail.com|
|     C00030|     Holly|  Collins|   cfuller@gmail.com|   Port Coltonton|   AL|gmail.com|
|     C000

In [0]:
df = df.withColumn("fullname", concat(col("first_name"), lit(" "), col("last_name")))
df.show(5)
df = df.drop("first_name", "last_name")
df.show(5)

+-----------+----------+---------+--------------------+------------------+-----+-------------+--------------+
|customer_id|first_name|last_name|               email|              city|state|       domain|      fullname|
+-----------+----------+---------+--------------------+------------------+-----+-------------+--------------+
|     C00001|     Emily|   Mooney|   rushjeff@ryan.org|      Johnsonmouth|   MS|     ryan.org|  Emily Mooney|
|     C00002|    Andrea|  Sellers|mccoykiara@kelly.com|       Stephenfort|   WY|    kelly.com|Andrea Sellers|
|     C00003|     Craig|    Hayes|rebeccamiller@yah...|South Stephenshire|   LA|    yahoo.com|   Craig Hayes|
|     C00004|     Bryan|    Scott|lawrence05@campbe...|         Chrisland|   ND|campbell.info|   Bryan Scott|
|     C00005|      Sean|  Vasquez|  carrie45@yahoo.com|   East Dennistown|   RI|    yahoo.com|  Sean Vasquez|
+-----------+----------+---------+--------------------+------------------+-----+-------------+--------------+
only showi

In [0]:
df.write.mode("overwrite").format("delta").save("abfss://silver@strdatabricks.dfs.core.windows.net/customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datapipeline.silver.customers
USING DELTA
LOCATION 'abfss://silver@strdatabricks.dfs.core.windows.net/customers'